In [1]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from idtxl.multivariate_pid import MultivariatePID
from idtxl.data import Data
import pandas as pd


def integerize_matrix(matrix):
    # 计算Z-Score
    zscore_matrix = (matrix - np.mean(matrix)) / np.std(matrix)
    
    # 取整
    integerized_matrix = np.round(zscore_matrix).astype(int)
    
    return integerized_matrix

def binaryize_matrix(matrix):
    # 计算Z-Score
    zscore_matrix = (matrix - np.mean(matrix)) / np.std(matrix)
    
    # 取整并转换为0和1
    integerized_matrix = np.where(zscore_matrix > 0, 1, 0)
    
    return integerized_matrix

In [2]:
monney_data_Lh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_dataset2.mat')['monney_timecourse_lh']
monney_data_Rh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_dataset2.mat')['monney_timecourse_rh']


normal_data_Lh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_dataset2.mat')['normal_timecourse_lh']
normal_data_Rh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_dataset2.mat')['normal_timecourse_rh']

#monney_data_Lh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_max.mat')['monney_timecourse_lh']
#monney_data_Rh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_max.mat')['monney_timecourse_rh']


#normal_data_Lh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_max.mat')['normal_timecourse_lh']
#normal_data_Rh=sio.loadmat('source_timecourse_Noise_-0.25_0_Signal_0.1_0.35_max.mat')['normal_timecourse_rh']

In [4]:
%%capture
#ROIs=['_V1_','_PIT_','_FFC_','_STSvp_','_STSva_']
#dataset2
#    '_V1_   '
#    '_V2_   '
#    '_V3_   '
#    '_V4_   '
#    '_PIT_  '00
#    '_FFC_  '
#    '_VVC_  '
#    '_STSvp_'
#    '_STSva_'
#    'TF     '

source = 'RV1LOFA'
target = 'LFFA'
save_name = 'synergy_result_list_' +source+ '_to_' +target +'_50ms_noresample.mat'
monney_data_s1=monney_data_Rh[0,:,:]
monney_data_s2=monney_data_Lh[4,:,:]
monney_data_t=monney_data_Lh[5,:,:]

normal_data_s1=normal_data_Rh[0,:,:]
normal_data_s2=normal_data_Lh[4,:,:]
normal_data_t=normal_data_Lh[5,:,:]

Monney_synergy=[];
Normal_synergy=[];
Monney_uni_s1=[];
Monney_uni_s2=[];

Normal_uni_s1=[];
Normal_uni_s2=[];

Monney_share_s12=[];
Normal_share_s12=[];
for i in range(21):
    data_source=[monney_data_s1[i],monney_data_s2[i]]
    data_target =monney_data_t[i]
    data_matrix1 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
    
    data_source=[normal_data_s1[i],normal_data_s2[i]]
    data_target =normal_data_t[i]
    data_matrix2 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
    print(i)
    for t_min in range(0,4140,1):
        t_max = t_min+60;



        data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)
        pid = MultivariatePID()
        settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}

        results_SxPID = pid.analyse_single_target(
        settings=settings_SxPID, data=data, target=0, sources=[1, 2])
        Monney_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
        Monney_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
        Monney_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
        Monney_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
    
    
    

            
        data_matrix = integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))
        data = Data(data_matrix2[:,:,t_min:t_max] , 'rps', normalise=False)
        pid = MultivariatePID()
        settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}

        results_SxPID = pid.analyse_single_target(
        settings=settings_SxPID, data=data, target=0, sources=[1, 2])
        Normal_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
        Normal_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
        Normal_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
        Normal_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
       
        sio.savemat(save_name, {'Monney_synergy': Monney_synergy, 'Normal_synergy': Normal_synergy,'Monney_uni_s1':Monney_uni_s1,'Monney_uni_s2':Monney_uni_s2,'Normal_uni_s1':Normal_uni_s1,'Normal_uni_s2':Normal_uni_s2,'Monney_share_s12':Monney_share_s12,'Normal_share_s12':Normal_share_s12})



In [9]:
np.shape(data_matrix2)

(1, 3, 4200)

In [36]:
  data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)

Adding data with properties: 3 processes, 60 samples, 1 replications
overwriting existing data


In [9]:
1

1

In [3]:
%%capture
#ROIs=['_V1_','_PIT_','_FFC_','_STSvp_','_STSva_']
#dataset2
#    '_V1_   '
#    '_V2_   '
#    '_V3_   '
#    '_V4_   '
#    '_PIT_  '
#    '_FFC_  '
#    '_VVC_  '
#    '_STSvp_'
#    '_STSva_'
#    'TF     '

source = 'LV1ROFA'
target = 'RFFA'
save_name = 'synergy_result_list_' +source+ '_to_' +target +'_50ms_noresample.mat'
monney_data_s1=monney_data_Lh[0,:,:]
monney_data_s2=monney_data_Rh[4,:,:]
monney_data_t=monney_data_Rh[5,:,:]

normal_data_s1=normal_data_Lh[0,:,:]
normal_data_s2=normal_data_Rh[4,:,:]
normal_data_t=normal_data_Rh[5,:,:]

Monney_synergy=[];
Normal_synergy=[];
Monney_uni_s1=[];
Monney_uni_s2=[];

Normal_uni_s1=[];
Normal_uni_s2=[];

Monney_share_s12=[];
Normal_share_s12=[];
for i in range(21):
    data_source=[monney_data_s1[i],monney_data_s2[i]]
    data_target =monney_data_t[i]
    data_matrix1 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
    
    data_source=[normal_data_s1[i],normal_data_s2[i]]
    data_target =normal_data_t[i]
    data_matrix2 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
    print(i)
    for t_min in range(0,4140,1):
        t_max = t_min+60;



        data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)
        pid = MultivariatePID()
        settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}

        results_SxPID = pid.analyse_single_target(
        settings=settings_SxPID, data=data, target=0, sources=[1, 2])
        Monney_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
        Monney_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
        Monney_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
        Monney_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
    
    
    

            
        data_matrix = integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))
        data = Data(data_matrix2[:,:,t_min:t_max] , 'rps', normalise=False)
        pid = MultivariatePID()
        settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}

        results_SxPID = pid.analyse_single_target(
        settings=settings_SxPID, data=data, target=0, sources=[1, 2])
        Normal_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
        Normal_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
        Normal_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
        Normal_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
       
        sio.savemat(save_name, {'Monney_synergy': Monney_synergy, 'Normal_synergy': Normal_synergy,'Monney_uni_s1':Monney_uni_s1,'Monney_uni_s2':Monney_uni_s2,'Normal_uni_s1':Normal_uni_s1,'Normal_uni_s2':Normal_uni_s2,'Monney_share_s12':Monney_share_s12,'Normal_share_s12':Normal_share_s12})



In [19]:
1

1

In [3]:
%%capture
source = 'LFFARFFA'
for target in ['LOFA','ROFA']:
    save_name = 'synergy_result_list_' +source+ '_to_' +target +'_50ms_noresample.mat'
    if target == 'LSTSvp':
        monney_data_s1=monney_data_Lh[5,:,:]
        monney_data_s2=monney_data_Rh[5,:,:]
        monney_data_t=monney_data_Lh[4,:,:]
        
        normal_data_s1=normal_data_Lh[5,:,:]
        normal_data_s2=normal_data_Rh[5,:,:]
        normal_data_t=normal_data_Lh[4,:,:]
    else:
        monney_data_s1=monney_data_Lh[5,:,:]
        monney_data_s2=monney_data_Rh[5,:,:]
        monney_data_t=monney_data_Rh[4,:,:]
        
        normal_data_s1=normal_data_Lh[5,:,:]
        normal_data_s2=normal_data_Rh[5,:,:]
        normal_data_t=normal_data_Rh[4,:,:]        
    Monney_synergy=[];
    Normal_synergy=[];
    Monney_uni_s1=[];
    Monney_uni_s2=[];
    
    Normal_uni_s1=[];
    Normal_uni_s2=[];
    
    Monney_share_s12=[];
    Normal_share_s12=[];
    for i in range(21):
        data_source=[monney_data_s1[i],monney_data_s2[i]]
        data_target =monney_data_t[i]
        data_matrix1 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        
        data_source=[normal_data_s1[i],normal_data_s2[i]]
        data_target =normal_data_t[i]
        data_matrix2 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        print(i)
        for t_min in range(0,4140,1):
            t_max = t_min+60;
    
    
    
            data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Monney_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Monney_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Monney_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Monney_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
        
        
        
    
                
            data_matrix = integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))
            data = Data(data_matrix2[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Normal_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Normal_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Normal_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Normal_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
           
            sio.savemat(save_name, {'Monney_synergy': Monney_synergy, 'Normal_synergy': Normal_synergy,'Monney_uni_s1':Monney_uni_s1,'Monney_uni_s2':Monney_uni_s2,'Normal_uni_s1':Normal_uni_s1,'Normal_uni_s2':Normal_uni_s2,'Monney_share_s12':Monney_share_s12,'Normal_share_s12':Normal_share_s12})



In [6]:
%%capture
source = 'LSTSvpRSTSvp'
for target in ['LV1','RV1']:
    save_name = 'synergy_result_list_' +source+ '_to_' +target +'_50ms_noresample.mat'
    if target == 'LV1':
        monney_data_s1=monney_data_Lh[7,:,:]
        monney_data_s2=monney_data_Rh[7,:,:]
        monney_data_t=monney_data_Lh[0,:,:]
        
        normal_data_s1=normal_data_Lh[7,:,:]
        normal_data_s2=normal_data_Rh[7,:,:]
        normal_data_t=normal_data_Lh[0,:,:]
    else:
        monney_data_s1=monney_data_Lh[7,:,:]
        monney_data_s2=monney_data_Rh[7,:,:]
        monney_data_t=monney_data_Rh[0,:,:]
        
        normal_data_s1=normal_data_Lh[7,:,:]
        normal_data_s2=normal_data_Rh[7,:,:]
        normal_data_t=normal_data_Rh[0,:,:]        
    Monney_synergy=[];
    Normal_synergy=[];
    Monney_uni_s1=[];
    Monney_uni_s2=[];
    
    Normal_uni_s1=[];
    Normal_uni_s2=[];
    
    Monney_share_s12=[];
    Normal_share_s12=[];
    for i in range(21):
        data_source=[monney_data_s1[i],monney_data_s2[i]]
        data_target =monney_data_t[i]
        data_matrix1 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        
        data_source=[normal_data_s1[i],normal_data_s2[i]]
        data_target =normal_data_t[i]
        data_matrix2 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        print(i)
        for t_min in range(0,4140,1):
            t_max = t_min+60;
    
    
    
            data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Monney_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Monney_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Monney_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Monney_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
        
        
        
    
                
            data_matrix = integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))
            data = Data(data_matrix2[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Normal_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Normal_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Normal_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Normal_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
           
            sio.savemat(save_name, {'Monney_synergy': Monney_synergy, 'Normal_synergy': Normal_synergy,'Monney_uni_s1':Monney_uni_s1,'Monney_uni_s2':Monney_uni_s2,'Normal_uni_s1':Normal_uni_s1,'Normal_uni_s2':Normal_uni_s2,'Monney_share_s12':Monney_share_s12,'Normal_share_s12':Normal_share_s12})


In [8]:
%%capture
source = 'LFFARFFA'
for target in ['LSTSvp','RSTSvp']:
    save_name = 'synergy_result_list_' +source+ '_to_' +target +'_50ms_noresample.mat'
    if target == 'LSTSvp':
        monney_data_s1=monney_data_Lh[5,:,:]
        monney_data_s2=monney_data_Rh[5,:,:]
        monney_data_t=monney_data_Lh[7,:,:]
        
        normal_data_s1=normal_data_Lh[5,:,:]
        normal_data_s2=normal_data_Rh[5,:,:]
        normal_data_t=normal_data_Lh[7,:,:]
    else:
        monney_data_s1=monney_data_Lh[5,:,:]
        monney_data_s2=monney_data_Rh[5,:,:]
        monney_data_t=monney_data_Rh[7,:,:]
        
        normal_data_s1=normal_data_Lh[5,:,:]
        normal_data_s2=normal_data_Rh[5,:,:]
        normal_data_t=normal_data_Rh[7,:,:]        
    Monney_synergy=[];
    Normal_synergy=[];
    Monney_uni_s1=[];
    Monney_uni_s2=[];
    
    Normal_uni_s1=[];
    Normal_uni_s2=[];
    
    Monney_share_s12=[];
    Normal_share_s12=[];
    for i in range(21):
        data_source=[monney_data_s1[i],monney_data_s2[i]]
        data_target =monney_data_t[i]
        data_matrix1 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        
        data_source=[normal_data_s1[i],normal_data_s2[i]]
        data_target =normal_data_t[i]
        data_matrix2 = np.array([integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))])
        print(i)
        for t_min in range(0,4140,1):
            t_max = t_min+60;
    
    
    
            data = Data(data_matrix1[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Monney_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Monney_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Monney_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Monney_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
        
        
        
    
                
            data_matrix = integerize_matrix(np.vstack([ data_target[np.newaxis, :],data_source]))
            data = Data(data_matrix2[:,:,t_min:t_max] , 'rps', normalise=False)
            pid = MultivariatePID()
            settings_SxPID = {'pid_estimator': 'SxPID', 'lags_pid': [0, 0], 'verbose':False}
    
            results_SxPID = pid.analyse_single_target(
            settings=settings_SxPID, data=data, target=0, sources=[1, 2])
            Normal_synergy.append(results_SxPID.get_single_target(0)['avg'][((1,2),)][2])
            Normal_uni_s1.append(results_SxPID.get_single_target(0)['avg'][((1,),)][2])
            Normal_uni_s2.append(results_SxPID.get_single_target(0)['avg'][((2,),)][2])
            Normal_share_s12.append(results_SxPID.get_single_target(0)['avg'][((1,),(2,),)][2])
    
        sio.savemat(save_name, {'Monney_synergy': Monney_synergy, 'Normal_synergy': Normal_synergy,'Monney_uni_s1':Monney_uni_s1,'Monney_uni_s2':Monney_uni_s2,'Normal_uni_s1':Normal_uni_s1,'Normal_uni_s2':Normal_uni_s2,'Monney_share_s12':Monney_share_s12,'Normal_share_s12':Normal_share_s12})
